In [ ]:
# CoineX - Pipeline CNN amélioré (4 canaux, prétraitement HAUTE résolution + CNN 384x384)
# ============================================================================================
# STRATÉGIE :
#   - Prétraitement à la RÉSOLUTION ORIGINALE de chaque image (bords Sobel ultra-nets)
#   - Noyaux morphologiques scalés selon la taille d'image (action consistante)
#   - Réduction unique à 384x384 à la toute fin (anti-aliasing INTER_AREA)
#   - CNN à 384x384 = 2.25x plus de pixels que l'ancien 256x256
#
# Améliorations vs version précédente (basées sur Cours_Image.pdf) :
#   - Pipeline 4 canaux : Luminance + Saturation HLS + Sobel + Masque Otsu/Morpho
#   - Gaussian blur avant Sobel (Semaine 9 / étape 1 de Canny)
#   - Egalisation d'histogramme sur le canal V de HSV (Semaine 7)
#   - HLS stable via cv2.cvtColor (remplace la formule manuelle instable)
#   - Otsu + Ouverture/Fermeture morphologiques (Semaines 5-6, 10) -> 4e canal binaire
#   - Normalisation par-canal avec stats calculées sur le train (Semaine 7)
#   - Architecture CNN VGG-like (double conv par bloc, ~1.19M params)
#   - Augmentation forte : photométrique + géométrique fin (rotation libre +- 20deg)
#   - Loss Huber/SmoothL1 (au lieu de MSE pur)
#   - AdamW + Cosine Annealing avec warm restarts
#   - Mixed Precision (AMP) - ~1.7x plus rapide sur T4
#   - Gradient clipping
#   - Early stopping (patience 30)
#   - Statistiques de normalisation embarquées dans le checkpoint

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))
    if len(filenames) > 5:
        print(f'  ... ({len(filenames)} files)')


In [ ]:
import os
import json
import random

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch : {torch.__version__}")
print(f"OpenCV  : {cv2.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")


In [ ]:
# =============================================================================
# 1. PIPELINE DE PRÉTRAITEMENT - 4 canaux à RÉSOLUTION ORIGINALE
# =============================================================================
# Stratégie hybride pour optimum vitesse/qualité :
#   - Lecture + Gaussian + HSV equalize + Sobel + Otsu à PLEINE résolution
#     (bords ultra-nets, threshold optimal)
#   - Réduction unique à 384x384 (INTER_AREA = anti-aliasing)
#   - Morphologie APRÈS le downsampling (rapide, et c'est l'échelle qui compte
#     pour le CNN de toute façon)
#
# Performance : ~750ms par image (~2min pour les 140 images préchargées)

TARGET_SIZE = 384   # Résolution finale envoyée au CNN

def pretraiter_image_brut(chemin_image):
    """
    Pipeline complet, sans normalisation (le Dataset s'en charge).

    Étapes :
      1) Lecture à la résolution originale
      2) Pré-débruitage Gaussien (Semaine 9 - étape 1 de Canny)
      3) Égalisation d'histogramme sur V de HSV (Semaine 7)
      4) Calcul des 4 canaux à PLEINE résolution :
         Canal 0 : Luminance Y = 0.299R + 0.587G + 0.114B
         Canal 1 : Saturation HLS (cv2 - stable)
         Canal 2 : Magnitude Sobel (bords ultra-nets sur l'image originale)
         Canal 3 : Masque Otsu à pleine résolution (threshold optimal)
      5) Réduction à 384x384 (INTER_AREA = anti-aliasing)
      6) Morphologie Open+Close à 384x384 sur le masque binaire (Semaine 10)

    Retourne : torch.Tensor (4, 384, 384), valeurs dans [0, 1].
    """
    img_bgr = cv2.imread(chemin_image)
    if img_bgr is None:
        raise FileNotFoundError(f"Impossible de lire l'image : {chemin_image}")

    # ----- 1) Pré-débruitage Gaussien (Semaine 9) -----
    # Cible le bruit pixel-par-pixel (capteur, JPEG) à l'échelle absolue.
    img_bgr = cv2.GaussianBlur(img_bgr, (5, 5), sigmaX=1.0)

    # ----- 2) Égalisation V de HSV (Semaine 7) -----
    # Robustesse aux changements d'éclairage entre photos.
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv[:, :, 2] = cv2.equalizeHist(hsv[:, :, 2])
    img_bgr_eq = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    img_rgb = cv2.cvtColor(img_bgr_eq, cv2.COLOR_BGR2RGB)

    # ----- Canal 0 : Luminance Y (Semaine 4) -----
    rgb_f = img_rgb.astype(np.float32) / 255.0
    gray = (0.299 * rgb_f[:, :, 0] + 0.587 * rgb_f[:, :, 1] + 0.114 * rgb_f[:, :, 2])

    # ----- Canal 1 : Saturation HLS (cv2 - stable) -----
    hls = cv2.cvtColor(img_bgr_eq, cv2.COLOR_BGR2HLS)
    sat = hls[:, :, 2].astype(np.float32) / 255.0

    # ----- Canal 2 : Magnitude Sobel à PLEINE résolution (Semaine 9) -----
    # ksize=3 reste optimal (opérateur de dérivée scale-invariant).
    # C'est la résolution de l'IMAGE qui donne la finesse, pas le noyau.
    gray_u8 = (gray * 255.0).astype(np.uint8)
    sobel_x = cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobel_x * sobel_x + sobel_y * sobel_y)
    sm_max = float(sobel_mag.max())
    if sm_max > 1e-6:
        sobel_mag = sobel_mag / sm_max

    # ----- Canal 3 : Otsu à pleine résolution (Semaines 5-6) -----
    # On garde Otsu sur la haute résolution (histogramme + threshold précis)
    # MAIS on déplace la morphologie après le downsampling pour la vitesse.
    _, otsu_hi = cv2.threshold(gray_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Forcer 'pièces = 1' (foreground minoritaire en général)
    if (otsu_hi > 0).mean() > 0.5:
        otsu_hi = 255 - otsu_hi

    # ----- Réduction à 384x384 -----
    tgt = (TARGET_SIZE, TARGET_SIZE)
    gray = cv2.resize(gray, tgt, interpolation=cv2.INTER_AREA)
    sat = cv2.resize(sat, tgt, interpolation=cv2.INTER_AREA)
    sobel_mag = cv2.resize(sobel_mag, tgt, interpolation=cv2.INTER_AREA)
    # Otsu : downsample puis re-binarise pour récupérer un vrai masque binaire
    otsu_small = cv2.resize(otsu_hi, tgt, interpolation=cv2.INTER_AREA)
    _, otsu_bin = cv2.threshold(otsu_small, 127, 255, cv2.THRESH_BINARY)

    # ----- Morphologie à 384x384 (Semaine 10) -----
    # Noyaux 5 et 11 = ~1.3% et 2.9% de la largeur, cohérent avec coins ~30-50px
    # Ouverture (supprime bruit isolé) puis fermeture (bouche trous gravures)
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    otsu_bin = cv2.morphologyEx(otsu_bin, cv2.MORPH_OPEN, k_open)
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    otsu_bin = cv2.morphologyEx(otsu_bin, cv2.MORPH_CLOSE, k_close)
    otsu_mask = otsu_bin.astype(np.float32) / 255.0

    # Empilement C x H x W (format PyTorch)
    stacked = np.stack([gray, sat, sobel_mag, otsu_mask], axis=0).astype(np.float32)
    return torch.from_numpy(stacked)


In [ ]:
# =============================================================================
# 2. DATASET avec normalisation par-canal et AUGMENTATION FORTE
# =============================================================================
def calculer_statistiques_dataset(preloaded_data):
    """Moyenne et écart-type par canal sur les données pré-chargées (Semaine 7)."""
    sum_ = torch.zeros(4, dtype=torch.float64)
    sum_sq = torch.zeros(4, dtype=torch.float64)
    n = 0
    for x, _ in preloaded_data:
        flat = x.view(4, -1).to(torch.float64)
        sum_ += flat.sum(dim=1)
        sum_sq += (flat * flat).sum(dim=1)
        n += flat.shape[1]
    mean = sum_ / n
    var = sum_sq / n - mean * mean
    std = torch.sqrt(torch.clamp(var, min=1e-8))
    return mean.to(torch.float32), std.to(torch.float32)


class CoinDataset(Dataset):
    def __init__(self, dossier_images, fichier_json, augment=False):
        self.augment = augment
        self.mean = None
        self.std = None

        with open(fichier_json, 'r') as f:
            self.labels_dict = json.load(f)
        self.liste_images = sorted(list(self.labels_dict.keys()))

        # Pré-chargement RAM
        self.preloaded_data = []
        print(f"  Pré-chargement {len(self.liste_images)} images depuis '{dossier_images}'...")
        for nom_image in self.liste_images:
            chemin = os.path.join(dossier_images, nom_image)
            x = pretraiter_image_brut(chemin)
            y = float(self.labels_dict[nom_image])
            self.preloaded_data.append((x, y))

    def set_statistics(self, mean, std):
        self.mean = mean.view(-1, 1, 1)
        self.std = std.view(-1, 1, 1)

    def __len__(self):
        return len(self.liste_images)

    def __getitem__(self, idx):
        x_raw, y = self.preloaded_data[idx]
        x = x_raw.clone()

        if self.augment:
            x = self._augmenter(x)

        # Normalisation par-canal (Semaine 7) - APRES augmentation
        if self.mean is not None:
            x = (x - self.mean) / self.std

        return x, torch.tensor([y], dtype=torch.float32)

    def _augmenter(self, x):
        """
        Augmentation forte (x: 4 x H x W, valeurs [0, 1]).
        - Géométrique : flips, rot 90/180/270, rotation libre +-20deg
        - Photométrique sur canaux continus uniquement (Otsu canal 3 protégé)
        """
        # --- Géométrique (tous canaux ensemble) ---
        if random.random() < 0.5:
            x = torch.flip(x, dims=[2])               # flip H
        if random.random() < 0.5:
            x = torch.flip(x, dims=[1])               # flip V
        if random.random() < 0.5:
            x = torch.rot90(x, random.choice([1, 2, 3]), dims=[1, 2])

        if random.random() < 0.7:
            angle = random.uniform(-20.0, 20.0)
            x = TF.rotate(x.unsqueeze(0), angle, fill=0.0,
                          interpolation=TF.InterpolationMode.BILINEAR).squeeze(0)

        # --- Photométrique (luminance uniquement pour brightness/contrast) ---
        if random.random() < 0.5:
            shift = random.uniform(-0.15, 0.15)
            x[0] = (x[0] + shift).clamp(0.0, 1.0)

        if random.random() < 0.5:
            scale = random.uniform(0.8, 1.2)
            m = x[0].mean()
            x[0] = ((x[0] - m) * scale + m).clamp(0.0, 1.0)

        # Bruit Gaussien sur canaux continus (Semaine 9 - grain capteur)
        if random.random() < 0.5:
            noise = torch.randn(3, x.shape[1], x.shape[2]) * 0.02
            x[:3] = (x[:3] + noise).clamp(0.0, 1.0)

        # Flou Gaussien léger (simule défocus)
        if random.random() < 0.3:
            sigma = random.uniform(0.5, 1.5)
            x_blur = TF.gaussian_blur(x[:3].unsqueeze(0),
                                     kernel_size=[5, 5],
                                     sigma=[sigma, sigma]).squeeze(0)
            x[:3] = x_blur

        return x


In [ ]:
# =============================================================================
# 3. ARCHITECTURE CNN - VGG-like (double conv par bloc) - 4 canaux d'entrée
# =============================================================================
class CustomCNN(nn.Module):
    """
    CNN from scratch (Semaine 12) :
      - 4 blocs convolutifs (Conv-BN-ReLU x2 + MaxPool)
      - Global Average Pooling (robuste à la translation)
      - 2 FC avec Dropout
    Paramètres totaux : ~1.19M
    """
    def __init__(self, in_channels=4):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2),
            )

        self.b1 = block(in_channels, 32)   # 256 -> 128
        self.b2 = block(32, 64)            # 128 -> 64
        self.b3 = block(64, 128)           # 64 -> 32
        self.b4 = block(128, 256)          # 32 -> 16

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
# =============================================================================
# 4. ENTRAÎNEMENT - Huber loss + AdamW + Cosine restarts + AMP + Early stop
# =============================================================================
TRAIN_DIR = "/kaggle/input/datasets/djezirioussama/dataset/data/train"
TRAIN_JSON = "/kaggle/input/datasets/djezirioussama/dataset/data/train.json"
VAL_DIR = "/kaggle/input/datasets/djezirioussama/dataset/data/validation"
VAL_JSON = "/kaggle/input/datasets/djezirioussama/dataset/data/validation.json"

def entrainer_modele(epochs=200, batch_size=16, lr=1e-3, patience=30,
                     seed=42, chemin_sauvegarde="/kaggle/working/meilleur_modele_nn.pth"):
    # Reproductibilité
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # --- Périphérique ---
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"SUCCÈS : GPU détecté → {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("ATTENTION : Aucun GPU - entraînement sur CPU.")

    # --- [1/5] Chargement datasets ---
    print("\n[1/5] Chargement des datasets...")
    train_dataset = CoinDataset(TRAIN_DIR, TRAIN_JSON, augment=True)
    val_dataset = CoinDataset(VAL_DIR, VAL_JSON, augment=False)

    # --- [2/5] Statistiques de normalisation (sur le TRAIN uniquement) ---
    print("\n[2/5] Calcul des statistiques de normalisation par canal...")
    mean, std = calculer_statistiques_dataset(train_dataset.preloaded_data)
    print(f"  Mean : {[f'{m:.4f}' for m in mean.tolist()]}")
    print(f"  Std  : {[f'{s:.4f}' for s in std.tolist()]}")
    train_dataset.set_statistics(mean, std)
    val_dataset.set_statistics(mean, std)

    pin_mem = device.type == "cuda"
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                             pin_memory=pin_mem, drop_last=False, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                           pin_memory=pin_mem, num_workers=2)

    # --- [3/5] Modèle, loss, optimiseur ---
    print("\n[3/5] Initialisation du modèle...")
    model = CustomCNN(in_channels=4).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Paramètres : {n_params:,}")

    # Huber/SmoothL1 : robuste aux outliers ET dérivable en 0
    # (Cours Semaine 3 : MSE pénalise les outliers; MAE n'est pas dérivable en 0)
    criterion = nn.SmoothL1Loss(beta=1.0)

    # AdamW : meilleure régularisation L2 que Adam
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)

    # Cosine annealing avec warm restarts : aide à sortir des minima locaux
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6
    )

    # AMP : ~1.7x plus rapide sur T4
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(device='cuda', enabled=use_amp)

    # --- [4/5] Boucle d'entraînement ---
    best_val_mae = float('inf')
    best_epoch = 0
    patience_counter = 0

    print(f"\n[4/5] Entraînement (max {epochs} epochs, early stop patience={patience})...")
    print("-" * 78)

    verif_hw_affichee = False

    for epoch in range(1, epochs + 1):
        # ===== TRAIN =====
        model.train()
        train_loss_acc = 0.0
        train_mae_acc = 0.0

        for inputs, targets in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            if not verif_hw_affichee:
                print(f"  [VÉRIF HW] Entrée: {inputs.device} | Cible: {targets.device} "
                      f"| Modèle: {next(model.parameters()).device}\n")
                verif_hw_affichee = True

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                predictions = model(inputs)
                loss = criterion(predictions, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss_acc += loss.item() * inputs.size(0)
            train_mae_acc += torch.sum(torch.abs(predictions.detach() - targets)).item()

        scheduler.step()
        train_loss_acc /= len(train_dataset)
        train_mae_acc /= len(train_dataset)

        # ===== VAL =====
        model.eval()
        val_loss_acc = 0.0
        val_mae_acc = 0.0
        val_exact = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)

                with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                    predictions = model(inputs)
                    loss = criterion(predictions, targets)

                val_loss_acc += loss.item() * inputs.size(0)
                pred_rounded = torch.round(predictions.float()).clamp(min=0.0)
                val_mae_acc += torch.sum(torch.abs(pred_rounded - targets)).item()
                val_exact += (pred_rounded == targets).sum().item()

        val_loss_acc /= len(val_dataset)
        val_mae_acc /= len(val_dataset)
        val_exact_pct = 100.0 * val_exact / len(val_dataset)

        improved = val_mae_acc < best_val_mae - 1e-4
        if improved:
            best_val_mae = val_mae_acc
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'mean': mean.tolist(),
                'std': std.tolist(),
                'in_channels': 4,
                'val_mae': val_mae_acc,
                'val_exact_pct': val_exact_pct,
                'epoch': epoch,
            }, chemin_sauvegarde)
        else:
            patience_counter += 1

        if epoch == 1 or epoch % 5 == 0 or improved or patience_counter >= patience:
            lr_now = optimizer.param_groups[0]['lr']
            tag = "  <-- BEST" if improved else ""
            print(f"Epoch {epoch:03d}/{epochs:03d} | "
                  f"Train Loss: {train_loss_acc:.4f} | Train MAE: {train_mae_acc:.2f} | "
                  f"Val MAE: {val_mae_acc:.2f} | Val Exact: {val_exact_pct:4.1f}% | "
                  f"LR: {lr_now:.2e}{tag}")

        # Early stopping
        if patience_counter >= patience:
            print(f"\n>>> Early stopping après {patience} epochs sans amélioration.")
            break

    # --- [5/5] Résumé final ---
    print("-" * 78)
    print(f"[5/5] Entraînement terminé.")
    print(f"  Meilleure Val MAE (arrondie) : {best_val_mae:.2f} pièces (epoch {best_epoch})")
    print(f"  Modèle + stats sauvegardés dans : {chemin_sauvegarde}")
    print(f"  >>> Téléchargez ce fichier et placez-le à la racine du projet local.")
    return best_val_mae


In [ ]:
# =============================================================================
# 5. LANCEMENT
# =============================================================================
# 200 epochs max, early stop patience=30 => arrêt automatique si plateau
# Le checkpoint sauvegarde poids + stats de normalisation dans un seul fichier
# Téléchargez-le depuis /kaggle/working/meilleur_modele_nn.pth
best_mae = entrainer_modele(epochs=200, batch_size=16, lr=1e-3, patience=30)
print(f"\n=== Résumé : meilleure Val MAE = {best_mae:.2f} pièces ===")
